<pre>
- Monóxido de carbono   -> Unidade de medida de retorno kg/kg
</pre>

Unidade nativa: kg/kg-¹ (razão de mistura em massa / mass mixing ratio)
O que significa: Quilogramas do gás poluidor por quilograma de ar.

Nota de conversão: Na prática e em estudos de qualidade do ar, costuma-se converter kg/kg para fração em volume em partes por milhão (ppm) ou partes por bilhão (ppb) utilizando a massa molar do gás e do ar seco (≈28,96 g/mol) 

In [ ]:
import os, sys
from datetime import datetime
from pyspark.sql import functions as F

In [ ]:
sys.path.append(os.path.abspath(os.path.join('..')))
import copernicus_cds_utils as utils 

In [ ]:
spark = utils.get_spark_session("Monoxido_Carbono")

In [ ]:
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

In [ ]:
def convert_unit(df_monoxido_carbono):
    # Converte o valor do monoxido de carbodo de kg/kg-¹ para ppb (partes por bilhão)
    M_AR = 28.9644 # g/mol
    M_CO = 28.0101 # g/mol
    FATOR_CONVERSAO = (M_AR / M_CO) * 1e9  # ~ 1.03407e9

    drop_cols = ["valid_time", "pressure_level", "co"]

    df_monoxido_carbono_ppb = \
        (df_monoxido_carbono
            .withColumns({"data_medicao": F.col("valid_time").cast("date")
                        ,"indicador": F.lit("Poluição do ar - CO (ppb)") 
                        ,"valor": (F.col("co") * F.lit(FATOR_CONVERSAO)).cast("double")
                        ,"unidade_medida": F.lit("ppb")})
            .drop(*drop_cols)
    )
    return df_monoxido_carbono_ppb

In [ ]:
dataset      = "cams-global-reanalysis-eac4"
variable     = "carbon_monoxide"
ano          = 2025

years_process = range(1991,2027)

for ano in years_process:
    start = datetime(2026, 7, 29).now()
    
    print("Obter dados, converter e gravar para o ano : ",ano, " - ", start, end="" )

    ret_download = utils.recuperar_dados_EAC4(dataset, variable, ano)

    file_name = r"{DATA_PATH_ROOT}EAC4-poluicao\EAC4_{variable}_{ano}.nc".format(DATA_PATH_ROOT=DATA_PATH_ROOT, variable=variable, ano=ano)
    os.rename(ret_download, file_name)

    df_spark = utils.converter_netcdf4_Spark_DF(spark, file_name)

    # Converte os dados para ppb (partes do bilhão)
    df_monoxido_carbono_ppb = convert_unit(df_spark)

    csv_path      = r"{DATA_PATH_ROOT}\EAC4-poluicao\arquivos_csv\co".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
    csv_file_name = f"EAC4_co_{ano}.csv"

    utils.write_data_csv(df_monoxido_carbono_ppb, csv_path, csv_file_name)

    print(f" - completed", "\n")


In [ ]:
df_spark.printSchema()
df_spark.show(10, False)